In [1]:
import xgboost
print(xgboost.__version__)

3.4.1


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)  # so results are reproducible

In [3]:
stations = ["Howrah", "Bardhaman", "Durgapur", "Asansol", "Dhanbad"]

segments = [
    {"origin": "Howrah", "destination": "Bardhaman", "distance_km": 95, "sched_travel_min": 90},
    {"origin": "Bardhaman", "destination": "Durgapur", "distance_km": 55, "sched_travel_min": 50},
    {"origin": "Durgapur", "destination": "Asansol", "distance_km": 40, "sched_travel_min": 40},
    {"origin": "Asansol", "destination": "Dhanbad", "distance_km": 60, "sched_travel_min": 60},
]

num_trains = 8
train_ids = [f"T{100+i}" for i in range(num_trains)]

start_date = datetime(2024, 1, 1)
num_days = 90

# A few holiday dates within our 90-day window (adjust as you like)
holidays = [
    datetime(2024, 1, 15),
    datetime(2024, 1, 26),
    datetime(2024, 2, 14),
]

In [4]:
train_bias_min = {tid: np.random.normal(loc=5, scale=8) for tid in train_ids}
# loc=5 means trains average a small inherent delay tendency, scale=8 adds spread
# clip so no train has a wildly unrealistic negative bias
train_bias_min = {tid: max(-5, val) for tid, val in train_bias_min.items()}

train_bias_min

{'T100': 8.973713224089861,
 'T101': 3.8938855906305228,
 'T102': 10.18150830480554,
 'T103': 17.184238851264205,
 'T104': 3.1267730022133122,
 'T105': 3.1269043444065554,
 'T106': 17.63370252405913,
 'T107': 11.139477833223271}

In [5]:
def get_weather(date):
    # Simple simulation: ~15% chance of rain/fog, otherwise clear
    return np.random.choice(["Clear", "Rain", "Fog"], p=[0.75, 0.15, 0.10])

def get_congestion(hour, is_weekend):
    if is_weekend:
        return np.random.choice(["Low", "Medium", "High"], p=[0.5, 0.35, 0.15])
    if hour in [7, 8, 9, 17, 18, 19]:  # peak hours
        return np.random.choice(["Low", "Medium", "High"], p=[0.1, 0.3, 0.6])
    return np.random.choice(["Low", "Medium", "High"], p=[0.5, 0.35, 0.15])

def compute_delay(base_bias, weather, congestion, is_weekend, is_holiday, distance_km):
    delay = base_bias

    # Weather effect
    if weather == "Rain":
        delay += np.random.normal(10, 5)
    elif weather == "Fog":
        delay += np.random.normal(15, 7)

    # Congestion effect
    if congestion == "Medium":
        delay += np.random.normal(8, 4)
    elif congestion == "High":
        delay += np.random.normal(20, 8)

    # Weekend effect (slightly less delay)
    if is_weekend:
        delay -= np.random.uniform(2, 6)

    # Holiday effect (big spike)
    if is_holiday:
        delay += np.random.normal(25, 10)

    # Distance effect (longer segments = more variance)
    delay += np.random.normal(0, distance_km * 0.05)

    return max(0, delay)  # no negative delays

In [6]:
rows = []

for day_offset in range(num_days):
    current_date = start_date + timedelta(days=day_offset)
    is_weekend = current_date.weekday() >= 5  # Saturday=5, Sunday=6
    is_holiday = current_date in holidays
    day_of_week = current_date.strftime("%A")

    for tid in train_ids:
        # Each train has a base departure hour (spread across the day)
        base_hour = 5 + (train_ids.index(tid) * 2) % 18
        current_departure = current_date.replace(hour=base_hour, minute=0, second=0)

        cumulative_delay = 0  # delay carried forward across segments

        for seg in segments:
            weather = get_weather(current_date)
            congestion = get_congestion(current_departure.hour, is_weekend)

            segment_delay = compute_delay(
                base_bias=train_bias_min[tid],
                weather=weather,
                congestion=congestion,
                is_weekend=is_weekend,
                is_holiday=is_holiday,
                distance_km=seg["distance_km"]
            )

            scheduled_departure = current_departure
            scheduled_arrival = scheduled_departure + timedelta(minutes=seg["sched_travel_min"])

            actual_departure = scheduled_departure + timedelta(minutes=cumulative_delay)
            actual_arrival = scheduled_arrival + timedelta(minutes=cumulative_delay + segment_delay)

            rows.append({
                "train_id": tid,
                "date": current_date.strftime("%Y-%m-%d"),
                "day_of_week": day_of_week,
                "origin_station": seg["origin"],
                "destination_station": seg["destination"],
                "scheduled_departure": scheduled_departure.strftime("%H:%M"),
                "scheduled_arrival": scheduled_arrival.strftime("%H:%M"),
                "actual_departure": actual_departure.strftime("%H:%M"),
                "actual_arrival": actual_arrival.strftime("%H:%M"),
                "current_delay_min": round(cumulative_delay, 1),
                "distance_km": seg["distance_km"],
                "weather": weather,
                "congestion_level": congestion,
                "hour_of_day": scheduled_departure.hour,
                "is_weekend": is_weekend,
                "is_holiday": is_holiday,
                "future_delay_min": round(cumulative_delay + segment_delay, 1)  # this is our ML TARGET
            })

            # carry delay forward to next segment
            cumulative_delay += segment_delay
            current_departure = scheduled_arrival  # next segment starts at scheduled time

df = pd.DataFrame(rows)
print(df.shape)
df.head(10)

(2880, 17)


,train_id,date,day_of_week,origin_station,destination_station,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,current_delay_min,distance_km,weather,congestion_level,hour_of_day,is_weekend,is_holiday,future_delay_min
0,T100,2024-01-01,Monday,Howrah,Bardhaman,05:00,06:30,05:00,06:57,0.0,95,Clear,High,5,False,False,27.8
1,T100,2024-01-01,Monday,Bardhaman,Durgapur,06:30,07:20,06:57,07:57,27.8,55,Clear,Low,6,False,False,37.4
2,T100,2024-01-01,Monday,Durgapur,Asansol,07:20,08:00,07:57,08:44,37.4,40,Clear,Medium,7,False,False,44.7
3,T100,2024-01-01,Monday,Asansol,Dhanbad,08:00,09:00,08:44,10:07,44.7,60,Clear,Medium,8,False,False,67.4
4,T101,2024-01-01,Monday,Howrah,Bardhaman,07:00,08:30,07:00,08:49,0.0,95,Clear,High,7,False,False,19.5
5,T101,2024-01-01,Monday,Bardhaman,Durgapur,08:30,09:20,08:49,09:50,19.5,55,Clear,Medium,8,False,False,30.2
6,T101,2024-01-01,Monday,Durgapur,Asansol,09:20,10:00,09:50,10:51,30.2,40,Clear,High,9,False,False,51.7
7,T101,2024-01-01,Monday,Asansol,Dhanbad,10:00,11:00,10:51,12:09,51.7,60,Clear,High,10,False,False,69.6
8,T102,2024-01-01,Monday,Howrah,Bardhaman,09:00,10:30,09:00,10:44,0.0,95,Clear,High,9,False,False,14.1
9,T102,2024-01-01,Monday,Bardhaman,Durgapur,10:30,11:20,10:44,12:12,14.1,55,Fog,Medium,10,False,False,52.1


In [7]:
print(df['future_delay_min'].describe())
print("\nWeather distribution:\n", df['weather'].value_counts())
print("\nCongestion distribution:\n", df['congestion_level'].value_counts())
print("\nHoliday rows:", df['is_holiday'].sum())
print("Weekend rows:", df['is_weekend'].sum())

count    2880.00000
mean       50.81500
std        36.15901
min         0.00000
25%        23.67500
50%        44.00000
75%        70.60000
max       284.00000
Name: future_delay_min, dtype: float64

Weather distribution:
 weather
Clear    2155
Rain      428
Fog       297
Name: count, dtype: int64

Congestion distribution:
 congestion_level
Low       1200
Medium     911
High       769
Name: count, dtype: int64

Holiday rows: 96
Weekend rows: 800


In [8]:
def minutes_to_hms(minutes):
    total_seconds = int(round(minutes * 60))
    hours = total_seconds // 3600
    mins = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    return f"{hours}h {mins}m {secs}s"

df['future_delay_hms'] = df['future_delay_min'].apply(minutes_to_hms)
df[['future_delay_min', 'future_delay_hms']].head()

,future_delay_min,future_delay_hms
0,27.8,0h 27m 48s
1,37.4,0h 37m 24s
2,44.7,0h 44m 42s
3,67.4,1h 7m 24s
4,19.5,0h 19m 30s


In [9]:
df.to_csv("../data/historical_trains.csv", index=False)
print("Saved successfully!")

Saved successfully!
